## 环境准备

In [ ]:
import sys

sys.path.append("..")

In [ ]:
import json
from pathlib import Path
from typing import Literal

import jieba
import numpy as np
import pandas as pd

## 超参数

### 路径

In [ ]:
ROOT = Path.cwd().parent


def get_dir(dir: Path) -> Path:
    if not dir.exists():
        dir.mkdir(parents=True)
    return dir


DATASETS_DIR = get_dir(ROOT / "datasets")
OUTPUTS_DIR = get_dir(ROOT / "outputs")

RAW_DATASETS_DIR = get_dir(DATASETS_DIR / "raw")
PROCESSED_DATASETS_DIR = get_dir(DATASETS_DIR / "processed")

MODELS_DIR = get_dir(OUTPUTS_DIR / "models")
LOGS_DIR = get_dir(OUTPUTS_DIR / "logs")
CHECKPOINTS_DIR = get_dir(OUTPUTS_DIR / "checkpoints")
FIGURES_DIR = get_dir(OUTPUTS_DIR / "figures")

STOPWORDS_PATH = DATASETS_DIR / "stopwords.txt"
RAW_DATASETS_PATH = RAW_DATASETS_DIR / "online_shopping_10_cats.csv"
PROCESSED_DATASETS_PATH = PROCESSED_DATASETS_DIR / "processed_datasets.csv"

BEST_MODEL_PATH = MODELS_DIR / "best_model.pth"
LAST_MODEL_PATH = MODELS_DIR / "last_model.pth"

### 数据

In [ ]:
# 最长序列长度
MAX_SEQ_LENGTH = 128

# 最小频率
MIN_FREQ = 3

# 是否移除停用词
REMOVE_STOPWORDS = True

# 训练集, 验证集, 测试集的比例
TRAIN_RATIO = 0.7
VALID_RATIO = 0.2
TEST_RATIO = 0.1

### 模型

In [ ]:
# 模型类型
MODEL_TYPE: Literal["LSTM", "RNN", "GRU"] = "LSTM"

# 嵌入维度
EMBEDDING_DIM = 300

# 隐藏层维度
HIDDEN_DIM = 256

# 层数数量
NUM_LAYERS = 2

# Dropout率
DROPOUT = 0.5

# 是否使用双向RNN
BIDIRECTIONAL = True

# 使用Attention机制
USE_ATTENTION = True

### 训练

In [ ]:
# Batch大小
BATCH_SIZE = 64

# 学习率
LEARNING_RATE = 1e-3

# 权重衰减
WEIGHT_DECAY = 1e-5

# 训练轮数
EPOCHS = 50

# 梯度裁剪
GRAD_CLIP = 1.0

# 学习率调整参数
# 当验证集性能不再提升时，学习率乘以LR_REDUCE_FACTOR
LR_REDUCE_FACTOR = 0.5
# 当验证集性能不再提升时，等待LR_REDUCE_PATIENCE轮后才调整学习率
LR_REDUCE_PATIENCE = 3

# 早停参数
# 当验证集性能不再提升时，等待EARLY_STOP_PATIENCE轮后停止训练
EARLY_STOP_PATIENCE = 5
# 当验证集性能提升小于EARLY_STOP_MIN_DELTA时，认为性能不再提升
EARLY_STOP_MIN_DELTA = 1e-4

## 数据处理

### 词表映射

In [ ]:
class VocabMapping:
    def __init__(self, word2idx: dict[str, int], idx2word: dict[int, str]):
        self.word2idx = word2idx
        self.idx2word = idx2word

    @property
    def vocab_size(self) -> int:
        return len(self.word2idx)

    @property
    def pad_idx(self) -> int:
        return 0

    @property
    def unk_idx(self) -> int:
        return 1

    def to_dict(self) -> dict[str, dict]:
        return {
            "word2idx": self.word2idx,
            "idx2word": {str(k): v for k, v in self.idx2word.items()},
        }

    @classmethod
    def from_dict(cls, data: dict[str, dict]) -> "VocabMapping":
        word2idx = data["word2idx"]
        idx2word = {int(k): v for k, v in data["idx2word"].items()}
        return cls(word2idx, idx2word)

    def save(self, path: Path) -> None:
        with path.open("w", encoding="utf-8") as f:
            json.dump(self.to_dict(), f, ensure_ascii=False, indent=4)

    @classmethod
    def load(cls, path: Path) -> "VocabMapping":
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
            return cls.from_dict(data)

### 数据预处理

In [ ]:
df = pd.read_csv(RAW_DATASETS_PATH)
df

,Unnamed: 0,cat,label,text
0,0,书籍,1,﻿做父母一定要有刘墉这样的心态，不断地学习，不断地进步，不断地给自己补充新鲜血液，让自己保持...
1,1,书籍,1,作者真有英国人严谨的风格，提出观点、进行论述论证，尽管本人对物理学了解不深，但是仍然能感受到...
2,2,书籍,1,作者长篇大论借用详细报告数据处理工作和计算结果支持其新观点。为什么荷兰曾经县有欧洲最高的生产...
3,3,书籍,1,作者在战几时之前用了＂拥抱＂令人叫绝．日本如果没有战败，就有会有美军的占领，没胡官僚主义的延...
4,4,书籍,1,作者在少年时即喜阅读，能看出他精读了无数经典，因而他有一个庞大的内心世界。他的作品最难能可贵...
...,...,...,...,...
62769,62769,酒店,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...
62770,62770,酒店,0,房间很小，整体设施老化，和四星的差距很大。毛巾太破旧了。早餐很简陋。房间隔音很差，隔两间房间...
62771,62771,酒店,0,我感觉不行。。。性价比很差。不知道是银川都这样还是怎么的！
62772,62772,酒店,0,房间时间长，进去有点异味！服务员是不是不够用啊！我在一楼找了半个小时以上才找到自己房间，想找...


In [ ]:
def read_raw(filepath: Path = RAW_DATASETS_PATH) -> tuple[list[str], list[int]]:
    """
    读取原始数据集文件，返回一个包含文本和标签的列表。
    Args:
        filepath (Path): 原始数据集文件的路径
    Returns:
        texts (list[str]): 文本列表
        labels (list[int]): 标签列表
    """
    if filepath.suffix == ".csv":
        df = pd.read_csv(filepath)
    elif filepath.suffix == ".json":
        df = pd.read_json(filepath)
    else:
        raise ValueError(f"不支持的文件格式: {str(filepath)}")

    texts = df["text"].astype(str).str.strip().tolist()
    labels = df["label"].astype(int).tolist()

    # 过滤空文本和非法标签
    valid = [(t, l) for t, l in zip(texts, labels) if t and (l in (0, 1))]  # noqa: E741
    texts = [v[0] for v in valid]
    labels = [v[1] for v in valid]

    print(f"读取数据: {len(texts)}条")
    print(f"正样本: {sum(labels)}, 负样本: {len(labels) - sum(labels)}")

    return texts, labels


texts, labels = read_raw()
texts[:5], labels[:5]

读取数据: 62774条
正样本: 31728, 负样本: 31046


(['\ufeff做父母一定要有刘墉这样的心态，不断地学习，不断地进步，不断地给自己补充新鲜血液，让自己保持一颗年轻的心。我想，这是他能很好的和孩子沟通的一个重要因素。读刘墉的文章，总能让我看到一个快乐的平易近人的父亲，他始终站在和孩子同样的高度，给孩子创造着一个充满爱和自由的生活环境。很喜欢刘墉在字里行间流露出的做父母的那种小狡黠，让人总是忍俊不禁，父母和子女之间有时候也是一种战斗，武力争斗过于低级了，智力较量才更有趣味。所以，做父母的得加把劲了，老思想老观念注定会一败涂地，生命不息，学习不止。家庭教育，真的是乐在其中。',
  '作者真有英国人严谨的风格，提出观点、进行论述论证，尽管本人对物理学了解不深，但是仍然能感受到真理的火花。整本书的结构颇有特点，从当时（本书写于八十年代）流行的计算机话题引入，再用数学、物理学、宇宙学做必要的铺垫——这些内容占据了大部分篇幅，最后回到关键问题：电脑能不能代替人脑。和现在流行的观点相反，作者认为人的某种“洞察”是不能被算法模拟的。也许作者想说，人的灵魂是无可取代的。',
  '作者长篇大论借用详细报告数据处理工作和计算结果支持其新观点。为什么荷兰曾经县有欧洲最高的生产率？为什么在文化上有着深刻纽带关系的中国和日本却在经济发展上有着极大的差异？为什么英国的北美殖民地造就了经济强大的美国，而西班牙的北美殖民却造就了范后的墨西哥？……很有价值，但不包括【中国近代史专业】。',
  '作者在战几时之前用了＂拥抱＂令人叫绝．日本如果没有战败，就有会有美军的占领，没胡官僚主义的延续，没有战后的民发反思，没有～，就不会让日本成为一个经济强国．当然，美国人也给日本人带来了耻辱．对日中关系也造成了深远的影响．文中揭露了＂东京审判＂中很多鲜为人知的东西．让人惊醒．唉！中国人民对日本的了解是不是太少了．',
  '作者在少年时即喜阅读，能看出他精读了无数经典，因而他有一个庞大的内心世界。他的作品最难能可贵的有两点，一是他的理科知识不错，虽不能媲及罗素，但与理科知识很差的作家相比，他的文章可读性要强；其二是他人格和文风的朴实，不造作，不买弄，让人喜欢。读他的作品，犹如听一个好友和你谈心，常常唤起心中的强烈的共鸣。他的作品90年后的更好些。衷心祝愿周国平健康快乐，为世人写出更多好作品。'],
 [1, 1, 1, 1, 1])

### 分词

In [ ]:
def tokenize(texts: list[str], remove_stopwords: bool = REMOVE_STOPWORDS) -> list[str]:
    """
    jieba 分词, 去掉停用词
    Args:
        texts(list[str]): 文本列表
        remove_stopwords(bool): 是否移除停用词
    Returns:
        result: 分词结果

    """
    stopwords = set()
    stopwords_path = STOPWORDS_PATH
    if remove_stopwords and stopwords_path.exists():
        with stopwords_path.open(mode="r", encoding="utf-8") as f:
            stopwords = set(line.strip for line in f if line.strip())

    result = []
    for text in texts:
        words = jieba.lcut(text)
        words = [w.strip() for w in words if w.strip()]
        if remove_stopwords:
            words = [w for w in words if w not in stopwords]
        if words:
            result.append(words)

    print(f"分词成功: {len(result)}")
    print(f"平均长度为: {np.mean([len(r) for r in result])}")
    return result

### 获取DataLoader

### 使用真实数据

## 模型定义

### Bahdanau 加性注意力

### BaseRNN 基类

### 创建模型

## 训练组件

### 早停

### 保存和加载checkpoint

## 训练

### 训练一个epoch

### 评测一个epoch

### 完整流程

## 可视化

## 测试集评估

## 混淆矩阵

## ROC 曲线

## 推理